In [3]:

import os, sys, glob, importlib
from google.colab import userdata


!pip install -q kaggle wandb


os.environ["KAGGLE_API_TOKEN"]      = userdata.get("KAGGLE_API_TOKEN")
os.environ["WANDB_API_KEY"]   = userdata.get("WANDB_API_KEY")
TOKEN = userdata.get("GITHUB_TOKEN")

REPO = "facial-expression-recognition"
REPO_PATH = f"/content/{REPO}"
if not os.path.isdir(REPO_PATH):
    !git clone https://{TOKEN}@github.com/amama22/{REPO}.git {REPO_PATH}
os.chdir(REPO_PATH)
if "src" not in sys.path:
    sys.path.append("src")
print("cwd:", os.getcwd())

!mkdir -p data
if not glob.glob("data/**/fer2013.csv", recursive=True):
    !kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p data
    !cd data && unzip -o -q "*.zip" && tar -xzf fer2013.tar.gz
paths = glob.glob("data/**/fer2013.csv", recursive=True)
print("csv:", paths)


import torch, wandb
print("cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
wandb.login()


import data, models, train
importlib.reload(data); importlib.reload(models); importlib.reload(train)
print("imports OK | sweep_train:", hasattr(train, "sweep_train"),
      "| TinyCNN params:", models.count_params(models.TinyCNN()))

Cloning into '/content/facial-expression-recognition'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 58 (delta 27), reused 14 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 17.50 KiB | 8.75 MiB/s, done.
Resolving deltas: 100% (27/27), done.
cwd: /content/facial-expression-recognition
100% 285M/285M [00:01<00:00, 167MB/s]

csv: ['data/fer2013/fer2013.csv']
cuda: True | Tesla T4


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tasomamaladze123 (tasomamaladze123-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


imports OK | sweep_train: True | TinyCNN params: 37063


In [4]:
import importlib, models, train; importlib.reload(models); importlib.reload(train)
import wandb, functools
wandb.finish()

sweep_config = {
    "method": "bayes",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "arch":         {"value": "DeeperCNN"},
        "epochs":       {"value": 15},
        "optimizer":    {"values": ["adam", "sgd"]},
        "lr":           {"distribution": "log_uniform_values", "min": 2e-4, "max": 3e-3},
        "batch_size":   {"values": [64, 128]},
        "weight_decay": {"values": [0.0, 1e-4, 5e-4]},
        "augment":      {"value": True},
        "p_drop":       {"values": [0.2, 0.3, 0.4]},
        "scheduler":    {"values": ["cosine", "none"]},
    },
}
defaults = {"arch": "DeeperCNN", "epochs": 15, "optimizer": "adam", "lr": 1e-3,
            "batch_size": 64, "weight_decay": 1e-4, "augment": True, "p_drop": 0.3,
            "scheduler": "cosine", "seed": 42,
            "class_weighted_loss": False, "weighted_sampler": False}

sweep_id = wandb.sweep(sweep_config, project="fer2013")
wandb.agent(sweep_id, function=functools.partial(train.sweep_train, paths[0], defaults), count=8)

Create sweep with ID: w6q91wl5
Sweep URL: https://wandb.ai/tasomamaladze123-none/fer2013/sweeps/w6q91wl5


wandb: Agent Starting Run: 46iunme5 with config:
wandb: 	arch: DeeperCNN
wandb: 	augment: True
wandb: 	batch_size: 128
wandb: 	epochs: 15
wandb: 	lr: 0.0005787490866413427
wandb: 	optimizer: sgd
wandb: 	p_drop: 0.2
wandb: 	scheduler: none
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch 01  train_loss 1.819 acc 0.251 | val_loss 1.772 acc 0.284 f1 0.124
epoch 02  train_loss 1.737 acc 0.294 | val_loss 1.710 acc 0.309 f1 0.163
epoch 03  train_loss 1.686 acc 0.330 | val_loss 1.595 acc 0.376 f1 0.260
epoch 04  train_loss 1.626 acc 0.362 | val_loss 1.520 acc 0.418 f1 0.293
epoch 05  train_loss 1.575 acc 0.386 | val_loss 1.480 acc 0.433 f1 0.309
epoch 06  train_loss 1.530 acc 0.407 | val_loss 1.421 acc 0.455 f1 0.340
epoch 07  train_loss 1.489 acc 0.425 | val_loss 1.396 acc 0.463 f1 0.347
epoch 08  train_loss 1.464 acc 0.434 | val_loss 1.379 acc 0.460 f1 0.359
epoch 09  train_loss 1.436 acc 0.446 | val_loss 1.362 acc 0.480 f1 0.367
epoch 10  train_loss 1.411 acc 0.459 | val_loss 1.324 acc 0.486 f1 0.376
epoch 11  train_loss 1.399 acc 0.460 | val_loss 1.320 acc 0.494 f1 0.378
epoch 12  train_loss 1.378 acc 0.470 | val_loss 1.298 acc 0.500 f1 0.406
epoch 13  train_loss 1.368 acc 0.477 | val_loss 1.277 acc 0.511 f1 0.411
epoch 14  train_loss 1.351 acc 0.478 | val_loss 1.3

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▄▅▆▆▆▇▇▇▇███
train/loss,█▇▆▅▄▄▃▃▂▂▂▂▁▁▁
val/acc,▁▂▄▅▅▆▆▆▇▇▇▇███
val/loss,█▇▆▅▄▃▃▃▃▂▂▂▁▂▁
val/macro_f1,▁▂▄▅▅▆▆▇▇▇▇██▇█
best_val_acc,0.51714
epoch,15
lr,0.00058
test_acc,0.52438


wandb: Agent Starting Run: qj52g2lh with config:
wandb: 	arch: DeeperCNN
wandb: 	augment: True
wandb: 	batch_size: 64
wandb: 	epochs: 15
wandb: 	lr: 0.00032930625360206374
wandb: 	optimizer: adam
wandb: 	p_drop: 0.3
wandb: 	scheduler: cosine
wandb: 	weight_decay: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


...

epoch 01  train_loss 1.839 acc 0.238 | val_loss 1.852 acc 0.264 f1 0.095
epoch 02  train_loss 1.767 acc 0.281 | val_loss 1.811 acc 0.259 f1 0.080
epoch 03  train_loss 1.725 acc 0.305 | val_loss 1.727 acc 0.313 f1 0.165
epoch 04  train_loss 1.683 acc 0.329 | val_loss 1.669 acc 0.348 f1 0.217
epoch 05  train_loss 1.654 acc 0.346 | val_loss 1.643 acc 0.368 f1 0.234
epoch 06  train_loss 1.608 acc 0.369 | val_loss 1.526 acc 0.409 f1 0.281
epoch 07  train_loss 1.568 acc 0.386 | val_loss 1.468 acc 0.433 f1 0.299
epoch 08  train_loss 1.535 acc 0.400 | val_loss 1.444 acc 0.441 f1 0.307
epoch 09  train_loss 1.510 acc 0.414 | val_loss 1.429 acc 0.453 f1 0.321
epoch 10  train_loss 1.491 acc 0.422 | val_loss 1.393 acc 0.460 f1 0.332
epoch 11  train_loss 1.472 acc 0.429 | val_loss 1.414 acc 0.458 f1 0.323
epoch 12  train_loss 1.452 acc 0.439 | val_loss 1.391 acc 0.461 f1 0.333
epoch 13  train_loss 1.437 acc 0.445 | val_loss 1.372 acc 0.474 f1 0.343
epoch 14  train_loss 1.423 acc 0.449 | val_loss 1.3

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▄▄▅▆▆▇▇▇▇███
train/loss,█▇▆▅▅▄▄▃▃▂▂▂▁▁▁
val/acc,▁▁▃▄▄▆▆▆▇▇▇▇▇▇█
val/loss,█▇▆▆▅▄▃▃▂▂▂▂▂▂▁
val/macro_f1,▁▁▃▄▅▆▆▆▇▇▇▇▇▇█
best_val_acc,0.49067
epoch,15
lr,0.00055
test_acc,0.48816


wandb: Agent Starting Run: 7fo084vy with config:
wandb: 	arch: DeeperCNN
wandb: 	augment: True
wandb: 	batch_size: 64
wandb: 	epochs: 15
wandb: 	lr: 0.00028891593031210073
wandb: 	optimizer: sgd
wandb: 	p_drop: 0.2
wandb: 	scheduler: cosine
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch 01  train_loss 1.817 acc 0.251 | val_loss 1.762 acc 0.289 f1 0.132
epoch 02  train_loss 1.737 acc 0.298 | val_loss 1.680 acc 0.331 f1 0.191
epoch 03  train_loss 1.686 acc 0.325 | val_loss 1.609 acc 0.376 f1 0.257
epoch 04  train_loss 1.633 acc 0.357 | val_loss 1.548 acc 0.406 f1 0.282
epoch 05  train_loss 1.590 acc 0.376 | val_loss 1.513 acc 0.423 f1 0.302
epoch 06  train_loss 1.548 acc 0.395 | val_loss 1.453 acc 0.439 f1 0.327
epoch 07  train_loss 1.517 acc 0.411 | val_loss 1.421 acc 0.452 f1 0.335
epoch 08  train_loss 1.496 acc 0.423 | val_loss 1.402 acc 0.459 f1 0.346
epoch 09  train_loss 1.481 acc 0.425 | val_loss 1.400 acc 0.462 f1 0.347
epoch 10  train_loss 1.472 acc 0.432 | val_loss 1.383 acc 0.465 f1 0.353
epoch 11  train_loss 1.459 acc 0.435 | val_loss 1.389 acc 0.468 f1 0.354
epoch 12  train_loss 1.449 acc 0.443 | val_loss 1.376 acc 0.471 f1 0.359
epoch 13  train_loss 1.446 acc 0.445 | val_loss 1.384 acc 0.466 f1 0.352
epoch 14  train_loss 1.441 acc 0.442 | val_loss 1.3

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,███▇▇▆▆▅▄▃▃▂▂▁▁
train/acc,▁▃▄▅▆▆▇▇▇▇█████
train/loss,█▇▆▅▄▃▂▂▂▂▁▁▁▁▁
val/acc,▁▃▄▅▆▇▇████████
val/loss,█▇▅▄▃▂▂▁▁▁▁▁▁▁▁
val/macro_f1,▁▃▅▆▆▇▇████████
best_val_acc,0.47088
epoch,15
lr,0.0
test_acc,0.47896


In [5]:
import importlib, train; importlib.reload(train)
import wandb; wandb.finish()
base = dict(arch="SmallResNet", lr=0.0012, batch_size=64, epochs=30, optimizer="adam",
            weight_decay=5e-4, augment=True, p_drop=0.2, scheduler="cosine", seed=42,
            class_weighted_loss=False, weighted_sampler=False)
train.run_experiment(base, paths[0], project="fer2013", run_name="C_imb_unweighted")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch 01  train_loss 1.779 acc 0.265 | val_loss 1.726 acc 0.284 f1 0.197
epoch 02  train_loss 1.536 acc 0.392 | val_loss 1.602 acc 0.421 f1 0.246
epoch 03  train_loss 1.415 acc 0.449 | val_loss 1.577 acc 0.417 f1 0.351
epoch 04  train_loss 1.354 acc 0.481 | val_loss 1.374 acc 0.451 f1 0.326
epoch 05  train_loss 1.305 acc 0.501 | val_loss 1.306 acc 0.498 f1 0.380
epoch 06  train_loss 1.262 acc 0.519 | val_loss 1.356 acc 0.489 f1 0.380
epoch 07  train_loss 1.218 acc 0.537 | val_loss 1.241 acc 0.522 f1 0.408
epoch 08  train_loss 1.183 acc 0.549 | val_loss 1.225 acc 0.531 f1 0.450
epoch 09  train_loss 1.154 acc 0.563 | val_loss 1.210 acc 0.553 f1 0.427
epoch 10  train_loss 1.126 acc 0.572 | val_loss 1.204 acc 0.540 f1 0.479
epoch 11  train_loss 1.100 acc 0.581 | val_loss 1.162 acc 0.560 f1 0.481
epoch 12  train_loss 1.085 acc 0.590 | val_loss 1.104 acc 0.571 f1 0.499
epoch 13  train_loss 1.059 acc 0.595 | val_loss 1.136 acc 0.570 f1 0.522
epoch 14  train_loss 1.036 acc 0.611 | val_loss 1.0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/acc,▁▃▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████
train/loss,█▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val/acc,▁▄▃▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████████
val/loss,█▇▇▅▄▅▄▄▃▃▃▂▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁
val/macro_f1,▁▂▄▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
best_val_acc,0.65812
epoch,30
lr,0.0
test_acc,0.66537


{'best_val_acc': 0.6581220395653385,
 'test_acc': 0.6653663973251602,
 'test_macro_f1': 0.6263236518361344}

In [6]:
import wandb; wandb.finish()
base = dict(arch="SmallResNet", lr=0.0012, batch_size=64, epochs=30, optimizer="adam",
            weight_decay=5e-4, augment=True, p_drop=0.2, scheduler="cosine", seed=42,
            weighted_sampler=False)
train.run_experiment({**base, "class_weighted_loss": True}, paths[0],
                     project="fer2013", run_name="C_imb_weightedloss")

epoch 01  train_loss 1.947 acc 0.179 | val_loss 1.921 acc 0.220 f1 0.160
epoch 02  train_loss 1.853 acc 0.231 | val_loss 1.784 acc 0.190 f1 0.196
epoch 03  train_loss 1.748 acc 0.306 | val_loss 1.722 acc 0.319 f1 0.239
epoch 04  train_loss 1.637 acc 0.379 | val_loss 1.880 acc 0.313 f1 0.252
epoch 05  train_loss 1.572 acc 0.406 | val_loss 1.599 acc 0.343 f1 0.310
epoch 06  train_loss 1.507 acc 0.429 | val_loss 1.502 acc 0.381 f1 0.339
epoch 07  train_loss 1.458 acc 0.444 | val_loss 1.456 acc 0.473 f1 0.399
epoch 08  train_loss 1.400 acc 0.469 | val_loss 1.464 acc 0.481 f1 0.391
epoch 09  train_loss 1.360 acc 0.485 | val_loss 1.379 acc 0.449 f1 0.382
epoch 10  train_loss 1.323 acc 0.499 | val_loss 1.385 acc 0.466 f1 0.402
epoch 11  train_loss 1.291 acc 0.513 | val_loss 1.320 acc 0.472 f1 0.386
epoch 12  train_loss 1.251 acc 0.528 | val_loss 1.389 acc 0.507 f1 0.427
epoch 13  train_loss 1.210 acc 0.536 | val_loss 1.260 acc 0.519 f1 0.466
epoch 14  train_loss 1.184 acc 0.549 | val_loss 1.3

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/acc,▁▂▃▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇████████
train/loss,█▇▇▆▆▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
val/acc,▁▁▃▃▃▄▆▆▅▅▆▆▆▇▆▇▆▇▇█▇█████████
val/loss,█▇▆█▅▅▄▄▄▄▃▄▃▃▂▂▂▂▁▂▂▁▁▂▁▁▁▁▁▁
val/macro_f1,▁▂▂▂▃▄▅▅▅▅▅▅▆▇▆▆▆▇▇▇▇▇▇███████
best_val_acc,0.6258
epoch,30
lr,0.0
test_acc,0.64029


{'best_val_acc': 0.6258010587907495,
 'test_acc': 0.6402897743103929,
 'test_macro_f1': 0.6170108847989965}

In [7]:
import wandb; wandb.finish()
base = dict(arch="SmallResNet", lr=0.0012, batch_size=64, epochs=30, optimizer="adam",
            weight_decay=5e-4, augment=True, p_drop=0.2, scheduler="cosine", seed=42,
            class_weighted_loss=False)
train.run_experiment({**base, "weighted_sampler": True}, paths[0],
                     project="fer2013", run_name="C_imb_weightedsampler")

epoch 01  train_loss 1.883 acc 0.228 | val_loss 1.751 acc 0.319 f1 0.283
epoch 02  train_loss 1.573 acc 0.395 | val_loss 1.764 acc 0.309 f1 0.239
epoch 03  train_loss 1.399 acc 0.472 | val_loss 1.673 acc 0.351 f1 0.274
epoch 04  train_loss 1.341 acc 0.496 | val_loss 1.656 acc 0.378 f1 0.308
epoch 05  train_loss 1.284 acc 0.517 | val_loss 1.536 acc 0.441 f1 0.364
epoch 06  train_loss 1.225 acc 0.540 | val_loss 1.254 acc 0.521 f1 0.466
epoch 07  train_loss 1.177 acc 0.555 | val_loss 1.229 acc 0.516 f1 0.475
epoch 08  train_loss 1.152 acc 0.568 | val_loss 1.210 acc 0.542 f1 0.485
epoch 09  train_loss 1.093 acc 0.587 | val_loss 1.200 acc 0.546 f1 0.493
epoch 10  train_loss 1.053 acc 0.607 | val_loss 1.176 acc 0.560 f1 0.512
epoch 11  train_loss 1.013 acc 0.620 | val_loss 1.199 acc 0.551 f1 0.512
epoch 12  train_loss 0.996 acc 0.626 | val_loss 1.160 acc 0.571 f1 0.509
epoch 13  train_loss 0.978 acc 0.630 | val_loss 1.085 acc 0.599 f1 0.570
epoch 14  train_loss 0.934 acc 0.648 | val_loss 1.1

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/acc,▁▃▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████
train/loss,█▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val/acc,▁▁▂▂▄▅▅▆▆▆▆▆▇▇▇▇▆▇▇▇▇█████████
val/loss,██▇▇▆▄▃▃▃▃▃▃▂▂▂▂▃▂▁▂▂▁▁▁▁▁▁▁▁▁
val/macro_f1,▂▁▂▂▃▅▅▅▅▆▆▆▇▇▇▆▆▇▇▇▇█████████
best_val_acc,0.66063
epoch,30
lr,0.0
test_acc,0.68069


{'best_val_acc': 0.6606297018668152,
 'test_acc': 0.6806910002786292,
 'test_macro_f1': 0.676899605394591}